# VLM keyframes: clean_shot → timestamp 3 FPS → GPU quality → PE-Core → P3

Giữ decode NVDEC theo batch, một persistent worker cho toàn bộ lượt chạy. OmniShotCut và PE-Core được load lazy một lần, cùng giữ trên GPU qua các video; inference tuần tự theo stage, giải phóng model khi worker kết thúc. Vùng lấy mẫu tối đa 30 giây. P3 common/medoid/unique + MMR budget=None; loại trùng cosine >0.96 chỉ giữa hai clean shot kề nhau, không dedup nội bộ shot. Shot toàn mẫu đồng màu bị loại; shot còn tín hiệu dùng fallback nếu cần. File scene inclusive chỉ xuất sau quality, P3, dedup và JPEG; chỉ giữ shot có keyframe cuối. Timeline đọc PTS/cờ packet bằng demux, bỏ PTS discard và packet cuối có PTS trùng chỉ khi byte xác nhận AVC end-of-sequence/stream trong MP4/MOV. NVDEC kiểm tra số frame đầu ra; SimpleDecoder giới hạn batch cuối theo display timeline. Không decode CPU để kiểm tra discard. Chỉ codec/profile không hỗ trợ NVDEC mới chuyển một bản tạm sang H.264 bằng CPU decode + NVENC encode, kiểm tra số frame/PTS trước khi dùng.


In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys
from IPython.display import display

WORK = Path('/kaggle/working/vlm_pipeline')
WORK.mkdir(parents=True,exist_ok=True)
PYTHON = sys.executable
JOBS_ROOT = WORK/'jobs'
VIDEO_BATCH_SIZE = 1  # download N -> extract N sequentially -> folder uploads -> cleanup N
UPLOAD_WORKERS = 4  # concurrent video files within the current upload folder
if type(UPLOAD_WORKERS) is not int or UPLOAD_WORKERS < 1:
    raise ValueError("UPLOAD_WORKERS must be a positive integer")
if type(VIDEO_BATCH_SIZE) is not int or VIDEO_BATCH_SIZE < 1:
    raise ValueError('VIDEO_BATCH_SIZE must be a positive integer')
DATASET = 'V3C'  # V3C: SFTP; LHE: Drive
START_VIDEO_ID, END_VIDEO_ID = 1, 1  # Default: verify LHE01; change this range as needed.
DOWNLOAD_ROOT_FOLDER_ID = '1yPb3EQBPKfKtlP0PGGZHmKDF3pghnRCN'
UPLOAD_ROOT_FOLDER_ID = '11__IfRj1eLK4GZWEeNLs1SQK-DowEU1K'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
# V3C source configuration copied from extract_keyframe_v3c.ipynb.
RCLONE_SFTP_CONFIG_SOURCE = Path('/kaggle/input/datasets/khanho/sftp-rclone/rclone.conf')
SFTP_REMOTE = 'stfp'
SFTP_VIDEO_SUBDIR = 'videos'
DATASET_GROUPS = ['V3C2']
VIDEO_ID_RANGES = {'V3C1': (38, 39), 'V3C2': (7476, 7485), 'V3C3': (17236, 17237)}
VIDEO_ID_LIST = {}  # e.g. {'V3C2': [7497, 7615]}; overrides the range
if DATASET not in {'V3C', 'LHE'}:
    raise ValueError('DATASET must be V3C or LHE')
CONFIG = dict(
    pipeline_id='vlm-p3-quality-v2',
    source_root=str(WORK/'source'), weights_cache=str(WORK/'weights'),
    omni_weights=None, batch_size=16, decode_batch_size=8, sample_fps=3.0,
    max_scene_seconds=30.0, final_similarity_threshold=0.96,
    min_hist_bins=5, overwrite=False,
)
if 'mmr_max_similarity' in CONFIG or CONFIG['pipeline_id'] != 'vlm-p3-quality-v2':
    raise RuntimeError('Stale VLM notebook configuration; open the updated vlm_keyframes.ipynb')
print('VLM pipeline:',CONFIG['pipeline_id'], 'dataset:',DATASET,
      'selection:', (DATASET_GROUPS, VIDEO_ID_RANGES, VIDEO_ID_LIST) if DATASET == 'V3C' else (START_VIDEO_ID, END_VIDEO_ID),
      'final cosine threshold:',CONFIG['final_similarity_threshold'])

def run(command,env=None):
    command=[str(x) for x in command]
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,env=env,bufsize=1)
    bars={}
    for line in process.stdout:
        if re.search(r'\d+%\|',line):
            stage=line.split(':',1)[0]
            payload={'text/plain':line.rstrip()}
            if stage not in bars: bars[stage]=display(payload,raw=True,display_id=True)
            else: bars[stage].update(payload,raw=True)
        else: print(line,end='',flush=True)
    if process.wait(): raise RuntimeError(f'Command failed ({process.returncode}): {command}')


Cell code 2: cài trực tiếp vào Python của Kaggle. Dùng PyTorch/torchvision sẵn có, không ép Python 3.11 và không tạo venv.

In [ ]:
REQUIREMENTS = "# Keep Kaggle's installed scientific/PyTorch stack when requirements are satisfied.\nnumpy\ntorch\ntorchvision\npandas\nscipy\nscikit-learn\nPillow\ntimm==1.0.15\neinops>=0.8.1\nomegaconf==2.3.0\nhuggingface-hub>=0.28.1\nftfy\nregex\niopath==0.1.10\nPyYAML\ntqdm\nffmpeg-python==0.2.0\nPyNvVideoCodec>=2.0.2,<3\n"
packages=[line.strip() for line in REQUIREMENTS.splitlines() if line.strip() and not line.startswith('#')]
run([PYTHON,'-m','pip','install','--disable-pip-version-check',*packages])
# Kaggle normally includes OpenCV; install only if absent.
import importlib.util
if importlib.util.find_spec('cv2') is None:
    run([PYTHON,'-m','pip','install','opencv-python-headless'])
run([PYTHON,'-m','pip','install','--no-deps','--no-build-isolation',
     'git+https://github.com/facebookresearch/perception_models.git@3e352cca660658d4b5c90f42a7808b11469e4c66',
     'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git@40ecf12a0f6ace617680cf22b74da0bef4af3c21'])


Cell code 3: rclone; V3C liệt kê collection/videos/ID/file trên SFTP, LHE liệt kê Drive. Hỗ trợ MP4/MOV/MKV/AVI/WEBM/WMV/MPEG/TS/MXF và các đuôi trong VIDEO_EXTENSIONS; ưu tiên MP4. Chỉ tải một video khi bắt đầu xử lý; upload vẫn theo cấu hình Drive của VLM.
Upload trực tiếp keyframes/<ID>.zip, map_keyframes/<ID>.csv, scenes/<ID>.scenes.txt, embeddings/<ID>.pkl; không thêm tầng V3C/LHE.


In [ ]:
"""Kaggle rclone installer reused from the embedding notebook; needs WORK and run."""
import os
import shutil
import subprocess
import urllib.request
import zipfile

# downloads.rclone.org có thể trả 403 trên Kaggle. Ưu tiên binary sẵn có,
# sau đó apt của hệ thống; nếu apt không dùng được thì thử release GitHub.
rclone = shutil.which("rclone")
if rclone is None and shutil.which("apt-get") and os.geteuid() == 0:
    apt = shutil.which("apt-get")
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    try:
        install = subprocess.run([apt, "install", "-y", "-qq", "rclone"], env=apt_env,
                                 capture_output=True, text=True, timeout=300)
        if install.returncode != 0:
            subprocess.run([apt, "update", "-qq"], env=apt_env, check=True,
                           capture_output=True, text=True, timeout=300)
            run([apt, "install", "-y", "-qq", "rclone"], env=apt_env)
        rclone = shutil.which("rclone")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print("apt không cài được rclone:", exc)

if rclone is None:
    release = "v1.75.2"
    archive = f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive}"
    rclone_zip = WORK / archive
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "kaggle-rclone-setup"})
        with urllib.request.urlopen(request, timeout=120) as src, rclone_zip.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        with zipfile.ZipFile(rclone_zip) as zf:
            matches = [item for item in zf.infolist() if item.filename.endswith("/rclone") and not item.is_dir()]
            if len(matches) != 1:
                raise RuntimeError("Release rclone không chứa đúng một binary")
            binary = WORK / "rclone"
            with zf.open(matches[0]) as src, binary.open("wb") as dst:
                shutil.copyfileobj(src, dst)
        binary.chmod(0o755)
        rclone = str(binary)
    except Exception as exc:
        raise RuntimeError("Không cài được rclone qua apt hoặc GitHub release. "
                           "Hãy gắn rclone binary vào Kaggle và thêm nó vào PATH.") from exc
run([rclone, "version"])

BATCH_SOURCE = '"""V3C SFTP / LHE Drive download, ZIP, upload and isolated cleanup."""\nimport csv\nfrom pathlib import Path\nimport shutil\nimport zipfile\nimport subprocess\nfrom concurrent.futures import ThreadPoolExecutor\n\nV3C_LIMITS = {\'V3C1\': (1, 7475), \'V3C2\': (7476, 17235), \'V3C3\': (17236, 28450)}\nVIDEO_EXTENSIONS = frozenset({\n    \'.mp4\', \'.mov\', \'.mkv\', \'.avi\', \'.webm\', \'.wmv\', \'.asf\', \'.m4v\',\n    \'.mpg\', \'.mpeg\', \'.mpe\', \'.mpv\', \'.m1v\', \'.m2v\', \'.ts\', \'.mts\',\n    \'.m2ts\', \'.m2t\', \'.mxf\', \'.vob\', \'.flv\', \'.f4v\', \'.3gp\', \'.3g2\',\n    \'.ogv\', \'.ogg\', \'.rm\', \'.rmvb\', \'.dv\', \'.divx\', \'.qt\', \'.nut\', \'.nsv\', \'.amv\',\n})\nVIDEO_EXTENSION_PRIORITY = {ext: rank for rank, ext in enumerate((\'.mp4\', \'.mov\', \'.mkv\', \'.webm\', \'.avi\'))}\n\n\ndef select_sftp_jobs(rclone, config_path, remote, collections, ranges, id_lists=None,\n                     video_subdir=\'videos\', completed=(), command=None):\n    """V3C SFTP layout: collection/videos/00001/00001.<video extension>.\n\n    List in chunks of 200, fall back to individual folders, prefer MP4 then MOV.\n    Extensions discover candidates only; ffprobe/NVDEC check the actual codec.\n    Explicit ID lists override ranges; clamp selections to collection bounds.\n    Only metadata is listed here; download one source video at a time later.\n    """\n    command = command or subprocess.run\n    jobs = []\n    if not remote or \':\' in remote or video_subdir != \'videos\':\n        raise ValueError(\'Invalid SFTP remote or video subdirectory\')\n\n    def listing(*args):\n        return command([str(rclone), \'--config\', str(config_path), \'lsf\', *args],\n                       capture_output=True, text=True, check=False)\n\n    for collection in dict.fromkeys(collections):\n        if collection not in V3C_LIMITS:\n            raise ValueError(f\'Unknown V3C collection: {collection}\')\n        lower, upper = V3C_LIMITS[collection]\n        explicit = (id_lists or {}).get(collection)\n        if explicit:\n            wanted = sorted({int(n) for n in explicit if lower <= int(n) <= upper})\n        else:\n            first, last = map(int, ranges[collection])\n            if first > last:\n                raise ValueError(f\'Invalid range: {collection}\')\n            wanted = range(max(first, lower), min(last, upper) + 1)\n        ids = [f\'{n:05d}\' for n in wanted if f\'{collection}/{n:05d}\' not in completed]\n        root = f\'{collection}/{video_subdir}\'\n        by_id = {}\n        for start in range(0, len(ids), 200):\n            chunk = ids[start:start + 200]\n            include = \'/{\' + \',\'.join(chunk) + \'}/*\' if len(chunk) > 1 else f\'/{chunk[0]}/*\'\n            result = listing(\'-R\', \'--files-only\', \'--include\', include, f\'{remote}:{root}/\')\n            if result.returncode:\n                print(f\'{collection}: recursive listing failed; trying individual folders\')\n                continue\n            for entry in result.stdout.splitlines():\n                path = Path(entry.strip())\n                if len(path.parts) == 2 and path.parent.name in chunk:\n                    by_id.setdefault(path.parent.name, []).append(path.name)\n        for video_id in ids:\n            if video_id not in by_id:\n                result = listing(\'--files-only\', f\'{remote}:{root}/{video_id}/\')\n                if result.returncode:\n                    raise RuntimeError(f\'Cannot list SFTP folder {collection}/{video_id}: {result.stderr}\')\n                by_id[video_id] = [entry.strip() for entry in result.stdout.splitlines()]\n            names = [name for name in by_id[video_id]\n                     if Path(name).name == name and Path(name).stem == video_id\n                     and Path(name).suffix.lower() in VIDEO_EXTENSIONS]\n            if not names:\n                print(f\'{collection}/{video_id}: no matching video file; skipped. \'\n                      f\'Files seen: {by_id[video_id][:10]}\')\n                continue\n            filename = min(names, key=lambda n: (\n                VIDEO_EXTENSION_PRIORITY.get(Path(n).suffix.lower(), len(VIDEO_EXTENSION_PRIORITY)),\n                n.lower(), n))\n            jobs.append(dict(source_name=filename, video_id=video_id, group=\'V3C\',\n                             collection=collection, source_kind=\'sftp\',\n                             remote_path=f\'{root}/{video_id}/{filename}\'))\n    return jobs\n\n\ndef _valid_job(job):\n    name, video_id = job[\'source_name\'], job[\'video_id\']\n    if Path(name).name != name or Path(name).stem != video_id:\n        return False\n    if job[\'group\'] == \'LHE\':\n        return name in set(range_names(1, 75)) and job.get(\'source_kind\', \'drive\') == \'drive\'\n    if job[\'group\'] != \'V3C\' or job.get(\'source_kind\') != \'sftp\':\n        return False\n    collection = job.get(\'collection\')\n    if collection not in V3C_LIMITS or len(video_id) != 5 or not video_id.isascii() or not video_id.isdigit():\n        return False\n    lower, upper = V3C_LIMITS[collection]\n    return (lower <= int(video_id) <= upper and Path(name).suffix.lower() in VIDEO_EXTENSIONS\n            and job.get(\'remote_path\') == f\'{collection}/videos/{video_id}/{name}\')\n\n\ndef range_names(start_id, end_id):\n    if (type(start_id) is not int or type(end_id) is not int\n            or not 1 <= start_id <= end_id <= 75):\n        raise ValueError(\'Range must satisfy 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 75\')\n    return [f\'LHE{index:02d}.mp4\' for index in range(start_id, end_id + 1)]\n\n\ndef select_remote_jobs(entries, start_id, end_id):\n    """Require every requested file exactly once before starting any downloads."""\n    names = range_names(start_id, end_id)\n    selected = {name: [] for name in names}\n    for entry in entries:\n        name = entry.get(\'Name\')\n        if not entry.get(\'IsDir\', False) and name in selected:\n            if entry.get(\'Path\', name) != name:\n                raise ValueError(f\'Expected source files directly inside the Drive root: {name}\')\n            selected[name].append(entry)\n    jobs = []\n    for name in names:\n        matches = selected[name]\n        if len(matches) != 1:\n            raise ValueError(f\'Expected exactly one {name} in source Drive folder; found {len(matches)}\')\n        size = matches[0].get(\'Size\', 0)\n        if type(size) is not int or size <= 0:\n            raise ValueError(f\'Invalid source video size: {name}\')\n        jobs.append(dict(source_name=name, video_id=Path(name).stem, group=\'LHE\', source_bytes=size))\n    return jobs\n\n\ndef cleanup_video(work_dir, jobs_root):\n    """Delete only the isolated per-video workspace, never input/Drive/shared runtime."""\n    work_dir, jobs_root = Path(work_dir), Path(jobs_root).resolve()\n    resolved = work_dir.resolve()\n    relative = resolved.relative_to(jobs_root)\n    group, video_id = relative.parts if len(relative.parts) == 2 else (\'\', \'\')\n    valid_id = (video_id in {Path(name).stem for name in range_names(1, 75)} if group == \'LHE\'\n                else group == \'V3C\' and len(video_id) == 5 and video_id.isascii()\n                and video_id.isdigit() and 1 <= int(video_id) <= 28450)\n    if (work_dir.is_symlink() or work_dir.parent.is_symlink() or not valid_id\n            or resolved.parent != jobs_root/group):\n        raise ValueError(f\'Refusing cleanup outside the per-video workspace: {resolved}\')\n    shutil.rmtree(resolved)\n\n\ndef download_remote_video(job, jobs_root, rclone, config_path, remote,\n                          download_root_id, upload_root_id, run,\n                          sftp_config=None, sftp_remote=None):\n    """Download and validate one source; leave its isolated workspace in place."""\n    is_sftp = job.get(\'source_kind\') == \'sftp\'\n    if (not _valid_job(job) or not upload_root_id\n            or (not is_sftp and (not download_root_id or download_root_id == upload_root_id))\n            or (is_sftp and (not sftp_config or not sftp_remote or \':\' in sftp_remote))):\n        raise ValueError(\'Invalid source job or Drive folder configuration\')\n    remote = remote.rstrip(\':\')\n    if not remote or \':\' in remote:\n        raise ValueError(\'Invalid rclone remote\')\n    jobs_root = Path(jobs_root).resolve()\n    jobs_root.mkdir(parents=True, exist_ok=True)\n    work = jobs_root/job[\'group\']/job[\'video_id\']\n    if work.is_symlink() or work.parent.is_symlink() or work.resolve().parent != jobs_root/job[\'group\']:\n        raise ValueError(\'Video workspace must be inside jobs_root\')\n    work.mkdir(parents=True, exist_ok=True)\n    video = work/job[\'source_name\']\n    # rclone checks transfers and raises through run() on failure; existing downloads can resume.\n    if is_sftp:\n        run([rclone, \'--config\', sftp_config, \'copyto\', f"{sftp_remote}:{job[\'remote_path\']}", video,\n             \'--retries\', \'3\', \'--multi-thread-streams\', \'8\', \'--multi-thread-cutoff\', \'32M\'])\n    else:\n        run([rclone, \'--config\', config_path, \'--drive-root-folder-id\', download_root_id,\n             \'--checksum\', \'--retries\', \'3\', \'copyto\', f"{remote}:{job[\'source_name\']}", video])\n    if (not video.is_file() or video.stat().st_size == 0\n            or (not is_sftp and video.stat().st_size != job[\'source_bytes\'])):\n        raise ValueError(f\'Download missing or wrong size: {video}\')\n    return work, video\n\n\ndef extract_downloaded_video(job, prepared, config, extract):\n    work, video = prepared\n    output = work/\'output\'\n    video_config = dict(config, video=str(video), group=job[\'group\'], output_dir=str(output))\n    extract(video_config, work/\'config.json\')\n\n\ndef process_video_batches(records, video_batch_size, jobs_root, config, rclone, config_path, remote,\n                          download_root_id, upload_root_id, run, extract,\n                          sftp_config=None, sftp_remote=None, upload_workers=4):\n    """Download/extract N; upload parallel files in ordered folder phases.\n\n    Yield each successfully uploaded/cleaned job immediately so the notebook can\n    record partial batch completion. Failed/pending workspaces remain for retry.\n    """\n    if type(video_batch_size) is not int or video_batch_size < 1:\n        raise ValueError(\'VIDEO_BATCH_SIZE must be a positive integer\')\n    if type(upload_workers) is not int or upload_workers < 1:\n        raise ValueError(\'UPLOAD_WORKERS must be a positive integer\')\n    for start in range(0, len(records), video_batch_size):\n        batch = records[start:start + video_batch_size]\n        print(f\'Video batch {start // video_batch_size + 1}: {len(batch)} videos\', flush=True)\n        prepared = []\n        for job in batch:\n            print(f\'Download: {job["source_name"]}\', flush=True)\n            prepared.append(download_remote_video(job, jobs_root, rclone, config_path, remote,\n                download_root_id, upload_root_id, run, sftp_config, sftp_remote))\n        for job, workspace in zip(batch, prepared):\n            print(f\'Extract: {job["source_name"]}\', flush=True)\n            extract_downloaded_video(job, workspace, config, extract)\n        results = upload_processed_batch(batch, prepared, rclone, config_path, remote,\n                                         upload_root_id, run, upload_workers)\n        # All four phases must succeed before deleting even the first workspace.\n        for job, workspace, result in zip(batch, prepared, results):\n            cleanup_video(workspace[0], jobs_root)\n            yield job, result\n\n\ndef upload_processed_batch(batch, prepared, rclone, config_path, remote, root_folder_id, run, workers=4):\n    """ZIPs -> scenes -> embeddings -> maps; parallel copyto within each phase.\n\n    Wait for every transfer in a folder before starting the next one. Failure\n    leaves all local workspaces intact; checksums/retries support rerunning.\n    """\n    remote = remote.rstrip(\':\')\n    if not remote or \':\' in remote or not root_folder_id:\n        raise ValueError(\'Set a valid rclone remote and Drive root_folder_id\')\n    if type(workers) is not int or workers < 1 or len(batch) != len(prepared):\n        raise ValueError(\'Invalid upload workers or batch workspaces\')\n    packages = []\n    for job, (work, _) in zip(batch, prepared):\n        archive, mapping, scenes, embeddings, count = package_keyframes(\n            work/\'output\', work/\'archives\', job)\n        packages.append(dict(keyframes=archive, scenes=scenes, embeddings=embeddings,\n                             map_keyframes=mapping, count=count))\n    results = [dict(video=job[\'source_name\'], keyframes=pack[\'count\'], destinations=[])\n               for job, pack in zip(batch, packages)]\n    common = [rclone, \'--config\', config_path, \'--drive-root-folder-id\', root_folder_id,\n              \'--checksum\', \'--retries\', \'3\']\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        for folder, suffix in [(\'keyframes\', \'.zip\'), (\'scenes\', \'.scenes.txt\'),\n                               (\'embeddings\', \'.pkl\'), (\'map_keyframes\', \'.csv\')]:\n            print(f\'Upload folder {folder}: {len(batch)} files, up to {workers} parallel\', flush=True)\n            futures = []\n            for job, pack, result in zip(batch, packages, results):\n                destination = f\'{remote}:{folder}/{job["video_id"]}{suffix}\'\n                result[\'destinations\'].append(destination)\n                futures.append(pool.submit(run, [*common, \'copyto\', pack[folder], destination]))\n            errors = []\n            for future in futures:\n                try:\n                    future.result()\n                except Exception as exc:\n                    errors.append(exc)\n            if errors:\n                raise RuntimeError(f\'Upload folder {folder} failed ({len(errors)} files); \'\n                                   \'all local batch files retained\') from errors[0]\n    return results\n\n\ndef package_keyframes(output_dir, archive_dir, job):\n    """Flat JPG ZIP per video plus a separate canonical map; reject partial outputs."""\n    root = Path(output_dir)\n    group, video_id = Path(job[\'group\']), job[\'video_id\']\n    image_dir = root/\'keyframes\'/group/video_id\n    mapping = root/\'map_keyframes\'/group/f\'{video_id}.csv\'\n    with mapping.open(newline=\'\', encoding=\'utf-8\') as handle:\n        reader = csv.DictReader(handle)\n        required = {\'keyframe_id\', \'video_id\', \'frame_idx\', \'timestamp_sec\', \'fps\'}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(f\'Invalid keyframe map columns: {mapping}\')\n        rows = list(reader)\n    if not rows:\n        raise ValueError(f\'No keyframes selected: {mapping}\')\n    expected = [f\'{index:06d}.jpg\' for index in range(len(rows))]\n    if ([int(row[\'keyframe_id\']) for row in rows] != list(range(len(rows)))\n            or any(row[\'video_id\'] != video_id for row in rows)):\n        raise ValueError(f\'Keyframe IDs/video IDs do not match: {mapping}\')\n    frames = [int(row[\'frame_idx\']) for row in rows]\n    if frames != sorted(set(frames)) or frames[0] < 0:\n        raise ValueError(f\'Invalid source frame indexes: {mapping}\')\n    actual = sorted(path.name for path in image_dir.iterdir() if path.is_file())\n    if actual != expected or any((image_dir/name).stat().st_size == 0 for name in expected):\n        raise ValueError(f\'Images do not match keyframe map: {image_dir}\')\n    archive = Path(archive_dir)/\'keyframes\'/group/f\'{video_id}.zip\'\n    archive.parent.mkdir(parents=True, exist_ok=True)\n    temporary = archive.with_suffix(\'.zip.partial\')\n    try:\n        # JPEG is already compressed. STORE avoids spending CPU recompressing it.\n        with zipfile.ZipFile(temporary, \'w\', compression=zipfile.ZIP_STORED, allowZip64=True) as handle:\n            for name in expected:\n                handle.write(image_dir/name, arcname=name)\n        with zipfile.ZipFile(temporary) as handle:\n            if handle.namelist() != expected or handle.testzip() is not None:\n                raise ValueError(f\'Invalid keyframe ZIP: {temporary}\')\n        temporary.replace(archive)\n    finally:\n        temporary.unlink(missing_ok=True)\n    scenes = root/\'scenes\'/group/f\'{video_id}.scenes.txt\'\n    embeddings = root/\'embeddings\'/group/f\'{video_id}.pkl\'\n    for path in [scenes, embeddings]:\n        if not path.is_file() or path.stat().st_size == 0:\n            raise ValueError(f\'Missing pipeline cache: {path}\')\n    return archive, mapping, scenes, embeddings, len(rows)\n'
exec(BATCH_SOURCE)
RCLONE_CONFIG=WORK/'rclone.conf'
if RCLONE_CONFIG_SOURCE.is_file():
    shutil.copyfile(RCLONE_CONFIG_SOURCE,RCLONE_CONFIG)
else:
    secret=os.environ.get('RCLONE_CONFIG')
    if not secret:
        from kaggle_secrets import UserSecretsClient
        secret=UserSecretsClient().get_secret('RCLONE_CONFIG')
    RCLONE_CONFIG.write_text(secret,encoding='utf-8')
    del secret
RCLONE_CONFIG.chmod(0o600)
SFTP_CONFIG = None
if DATASET == 'V3C':
    if not RCLONE_SFTP_CONFIG_SOURCE.is_file():
        raise FileNotFoundError(f'Attach SFTP config: {RCLONE_SFTP_CONFIG_SOURCE}')
    SFTP_CONFIG = WORK/'sftp-rclone.conf'
    shutil.copyfile(RCLONE_SFTP_CONFIG_SOURCE, SFTP_CONFIG)
    SFTP_CONFIG.chmod(0o600)
    jobs = select_sftp_jobs(rclone, SFTP_CONFIG, SFTP_REMOTE, DATASET_GROUPS,
                            VIDEO_ID_RANGES, VIDEO_ID_LIST, SFTP_VIDEO_SUBDIR)
else:
    if DOWNLOAD_ROOT_FOLDER_ID == UPLOAD_ROOT_FOLDER_ID:
        raise ValueError('Source and output Drive folders must differ')
    entries=json.loads(subprocess.check_output([
        str(rclone),'--config',str(RCLONE_CONFIG),'--drive-root-folder-id',DOWNLOAD_ROOT_FOLDER_ID,
        'lsjson',RCLONE_REMOTE.rstrip(':')+':','--files-only','--max-depth','1'],text=True))
    jobs=select_remote_jobs(entries,START_VIDEO_ID,END_VIDEO_ID)
if not jobs:
    raise RuntimeError('No matching videos found; check dataset/ranges and remote folders')
print(f'Selected {len(jobs)} videos: {jobs[0]["source_name"]} ... {jobs[-1]["source_name"]}')


Cell code 4: cài source chính được nhúng trong notebook và kiểm tra CUDA/NVDEC imports. Không cần clone repo riêng.

In [ ]:
SOURCE_ZIP = '...'
import base64, io, zipfile
SOURCE_ROOT=WORK/'source'
SOURCE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP))) as bundle:
    for name in bundle.namelist():
        target=(SOURCE_ROOT/name).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()): raise ValueError('Invalid bundled path')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(bundle.read(name))
smoke=WORK/'smoke.py'
smoke.write_text("""
import sys, shutil, torch, torchvision, PyNvVideoCodec as nvc, omnishotcut
assert shutil.which('ffprobe'), 'ffprobe is required to read source timestamps'
assert torch.cuda.is_available(), 'Select T4 GPU'
torch.cuda.set_device(0)
assert torch.cuda.get_device_capability(0)[0]>=7, 'Use T4 or newer; P100 is unsupported'
import core.vision_encoder.pe as pe
assert 'PE-Core-L14-336' in pe.CLIP.available_configs()
print('Python:',sys.version.split()[0], 'GPU:',torch.cuda.get_device_name(0))
print('PyTorch:',torch.__version__,'PyNvVideoCodec:',nvc.__version__)
print('Imports OK; actual NVDEC is validated when opening the first video.')
""",encoding='utf-8')
run([PYTHON,smoke])


Cell code 5: tải VIDEO_BATCH_SIZE video, xử lý lần lượt, upload song song tối đa UPLOAD_WORKERS file trong keyframes → scenes → embeddings → map_keyframes. Chờ xong mỗi folder mới sang folder sau. Upload hết batch thành công mới xóa workspace; lỗi giữ nguyên toàn bộ batch.


In [ ]:
"""Notebook processing cell: one resident GPU worker across all video batches."""
sys.path.insert(0, str(SOURCE_ROOT))
from src.keyframe_extraction.worker_session import VlmWorkerSession

worker_path = SOURCE_ROOT/'vlm_worker.py'
bars = {}


def show_worker_line(line):
    if re.search(r'\d+%\|', line):
        stage = line.split(':', 1)[0]
        payload = {'text/plain': line.rstrip()}
        if stage not in bars:
            bars[stage] = display(payload, raw=True, display_id=True)
        else:
            bars[stage].update(payload, raw=True)
    else:
        print(line, end='', flush=True)


def extract_video(config, config_path):
    bars.clear()
    session.extract(config, config_path)


UPLOAD_DONE = globals().get('UPLOAD_DONE', set())
pending = [job for job in jobs if (job['group'], job['video_id']) not in UPLOAD_DONE]
try:
    if pending:
        with VlmWorkerSession(
                [PYTHON, '-u', worker_path, '--serve', '--source-root', SOURCE_ROOT],
                on_line=show_worker_line) as session:
            for job, result in process_video_batches(
                    pending, VIDEO_BATCH_SIZE, JOBS_ROOT, CONFIG, rclone, RCLONE_CONFIG, RCLONE_REMOTE,
                    DOWNLOAD_ROOT_FOLDER_ID, UPLOAD_ROOT_FOLDER_ID, run, extract_video,
                    sftp_config=SFTP_CONFIG, sftp_remote=SFTP_REMOTE, upload_workers=UPLOAD_WORKERS):
                UPLOAD_DONE.add((job['group'], job['video_id']))
                print(f'Uploaded {job["video_id"]}: {result["keyframes"]} images; local job removed.', flush=True)
except Exception:
    print(f'Batch stopped; unfinished jobs retained under {JOBS_ROOT}. Re-run this cell to retry.', flush=True)
    raise
print(f'Completed {len(UPLOAD_DONE.intersection((job["group"], job["video_id"]) for job in jobs))}/{len(jobs)}')
